# Camada Silver - Datafiniti + Amazon Electronics (Quévin)

Limpa as tabelas Bronze e grava as tabelas Silver em Delta, na base `silver` do grupo (antes estava numa base
própria, `silver_electronics`). A lógica de limpeza é a do Quévin; mudou o que a revisão do modelo pediu.

| Tabela Silver | 1 linha = | Usada no modelo em |
|---|---|---|
| `silver.amazon_avaliacoes` | 1 avaliação de um produto, com data entre junho e dezembro de 2017 | `fact_avaliacoes` (fonte Amazon) |
| `silver.amazon_produtos` | 1 produto da Amazon (eletrónica) avaliado nesse período | `dim_produto` |
| `silver.datafiniti_precos` | 1 produto × retalhista × dia, entre junho e dezembro de 2017 | `fact_precos`, `dim_retalhista` |
| `silver.datafiniti_produtos` | 1 produto da Datafiniti | `dim_produto`, `dim_categoria_produto` |
| `silver.datafiniti_amazon_ligacao` | 1 par produto Datafiniti ↔ asin da Amazon | `dim_produto` (`id_produto_comum`, `tipo_ligacao`) |

Regras do grupo aplicadas:
- **Intervalo comum:** 2017-06-01 a 2017-12-31, já na Silver (antes o corte ficava para a Gold).
- Sessão Spark em UTC: as datas são convertidas a partir de UTC.
- Retalhista em falta fica NULL (na Gold passa ao registo `-1` da `dim_retalhista`).
- Produtos que não são eletrónica (livros, filmes) ficam de fora.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - Datafiniti + Amazon (Quévin)") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")   # datas que não batem com o formato ficam NULL

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

# Os textos da Amazon e da Datafiniti trazem entidades HTML (&amp; em vez de &, etc.)
def limpa_texto(c):
    for de, para in [("&amp;", "&"), ("&quot;", '"'), ("&#39;", "'"), ("&gt;", ">"), ("&lt;", "<")]:
        c = F.regexp_replace(c, de, para)
    c = F.trim(c)
    return F.when(c == "", None).otherwise(c)

## Avaliações da Amazon

- Linhas repetidas (mesmo asin, cliente, nota e data) são removidas; quando o mesmo cliente dá duas notas ao mesmo produto
  no mesmo dia, fica uma (a mais alta).
- `unixReviewTime` está em segundos desde 1970 (sempre 00:00 UTC): passa a `DATE`.
- `overall` só toma os valores 1 a 5: passa a `INT` (`nota`).
- Só ficam as avaliações de junho a dezembro de 2017.

In [4]:
ratings = spark.table("bronze.amazon_ratings")
print("Avaliações na Bronze:", ratings.count())

avaliacoes = (ratings
    .select(F.trim("asin").alias("asin"), F.trim("reviewerID").alias("reviewer_id"),
            F.col("overall").cast("double").cast("int").alias("nota"),
            F.to_date(F.timestamp_seconds(F.col("unixReviewTime").cast("long"))).alias("data_avaliacao"))
    .filter(F.col("asin").isNotNull() & F.col("nota").between(1, 5))
    .filter(F.col("data_avaliacao").between(DATA_INICIO, DATA_FIM))
    .dropDuplicates())
print("Avaliações de jun-dez 2017 (sem linhas repetidas):", avaliacoes.count())

# O mesmo cliente às vezes avalia o mesmo produto duas vezes no mesmo dia, com notas diferentes:
# fica uma avaliação por cliente, produto e dia (a última nota, a mais alta em caso de empate na ordem do ficheiro)
w = Window.partitionBy("asin", "reviewer_id", "data_avaliacao").orderBy(F.desc("nota"))
repetidas = avaliacoes.count() - avaliacoes.select("asin", "reviewer_id", "data_avaliacao").distinct().count()
print("Cliente x produto x dia com mais de uma nota:", repetidas)
avaliacoes = avaliacoes.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
assert avaliacoes.count() == avaliacoes.select("asin", "reviewer_id", "data_avaliacao").distinct().count()

Avaliações na Bronze: 20994353
Avaliações de jun-dez 2017 (sem linhas repetidas): 1549833
Cliente x produto x dia com mais de uma nota: 93


## Produtos da Amazon

- Um registo por asin; só os produtos com avaliações no intervalo.
- Ficam de fora os produtos que não são eletrónica (`main_cat` = Books, Movies & TV, etc.), e as suas avaliações.
- `brand` vazia ou genérica passa a NULL; `main_cat` com HTML passa a NULL.
- `category` é uma lista: fica o caminho completo e o 2.º nível (`subcategoria`; o 1.º é sempre Electronics).
- `price` passa a número só quando é um preço único; `date` passa a `DATE`; `rank` passa a número.

In [5]:
meta = spark.table("bronze.amazon_meta")
asins_avaliados = avaliacoes.select("asin").distinct()
marca = limpa_texto(F.col("brand"))
main_cat = limpa_texto(F.col("main_cat"))

produtos_amz = (meta
    .filter(F.col("asin").isNotNull())
    .dropDuplicates(["asin"])
    .join(asins_avaliados, "asin", "left_semi")
    .select("asin",
            limpa_texto(F.col("title")).alias("nome"),
            F.when(F.lower(marca).isin("generic", "unknown", "n/a", "na", "none"), None).otherwise(marca).alias("marca"),
            F.when(main_cat.startswith("<"), None).otherwise(main_cat).alias("categoria_principal"),
            limpa_texto(F.element_at("category", 2)).alias("subcategoria"),
            limpa_texto(F.concat_ws(" > ", "category")).alias("caminho_categoria"),
            F.regexp_replace(F.regexp_extract("price", r"^\$([0-9,]+\.[0-9]{2})$", 1), ",", "").cast("decimal(10,2)").alias("preco_lista"),
            F.to_date(F.trim("date"), "MMMM d, yyyy").alias("data_lancamento"),
            F.regexp_replace(F.regexp_extract("rank", r"#([0-9,]+)", 1), ",", "").cast("int").alias("ranking_vendas")))

print("main_cat dos produtos avaliados:")
produtos_amz.groupBy("categoria_principal").count().orderBy(F.desc("count")).show(40, truncate=False)

main_cat dos produtos avaliados:
+----------------------------+-----+
|categoria_principal         |count|
+----------------------------+-----+
|Computers                   |56472|
|Camera & Photo              |28728|
|Home Audio & Theater        |25680|
|All Electronics             |22766|
|Cell Phones & Accessories   |14918|
|Car Electronics             |5496 |
|Sports & Outdoors           |2743 |
|Tools & Home Improvement    |2274 |
|Office Products             |2207 |
|Automotive                  |1701 |
|Amazon Home                 |1525 |
|GPS & Navigation            |1399 |
|Industrial & Scientific     |1219 |
|null                        |1201 |
|Musical Instruments         |1141 |
|Portable Audio & Accessories|830  |
|Amazon Devices              |630  |
|Arts, Crafts & Sewing       |453  |
|Health & Personal Care      |445  |
|Toys & Games                |340  |
|Video Games                 |214  |
|Baby                        |163  |
|All Beauty                  |150  |
|Book

In [6]:
# Categorias que não são eletrónica
# (só as que claramente não são; acessórios, escritório e casa podem ser eletrónica e ficam)
# Os nomes são os que aparecem nos dados (ver a tabela acima). Os produtos de outros departamentos
# (ex.: Sports & Outdoors, Toys & Games) ficam, porque no ficheiro de eletrónica são acessórios eletrónicos.
NAO_ELETRONICA = ["Books", "Movies & TV", "Grocery", "Pet Supplies", "Arts, Crafts & Sewing", "All Beauty"]
excluidos = produtos_amz.filter(F.col("categoria_principal").isin(NAO_ELETRONICA))
print("Produtos excluídos (não são eletrónica):", excluidos.count())
excluidos.groupBy("categoria_principal").count().show(truncate=False)

produtos_amz = produtos_amz.filter(~F.col("categoria_principal").isin(NAO_ELETRONICA) | F.col("categoria_principal").isNull())
avaliacoes = avaliacoes.join(excluidos.select("asin"), "asin", "left_anti")

assert produtos_amz.count() == produtos_amz.select("asin").distinct().count()
print("Produtos da Amazon:", produtos_amz.count(), "| avaliações:", avaliacoes.count())
gravar_silver(avaliacoes, "amazon_avaliacoes")
gravar_silver(produtos_amz, "amazon_produtos")

Produtos excluídos (não são eletrónica): 762
+---------------------+-----+
|categoria_principal  |count|
+---------------------+-----+
|Pet Supplies         |47   |
|Books                |98   |
|Arts, Crafts & Sewing|453  |
|All Beauty           |150  |
|Grocery              |12   |
|Movies & TV          |2    |
+---------------------+-----+

Produtos da Amazon: 172168 | avaliações: 1546461
silver.amazon_avaliacoes: 1546461 linhas
silver.amazon_produtos: 172168 linhas


## Preços da Datafiniti

- `prices.dateSeen` tem várias datas na mesma célula: cada data passa a uma linha; só ficam as de junho a dezembro de 2017.
- Ficam só os preços em USD de produtos novos (`condition` escrita de várias formas).
- O mesmo retalhista escrito de formas diferentes é normalizado.
- `prices.isSale` não é fiável: a promoção real é `preco_min < preco_max`, e o desconto é `desconto_pct`.
- **Grão produto × retalhista × dia:** quando há mais de um preço para a mesma combinação, fica o preço mínimo e máximo
  do dia (antes o `drop_duplicates` sobre todas as colunas não garantia o grão).

In [7]:
raw = spark.table("bronze.datafiniti_precos").drop("_data_ingestao", "_ficheiro_origem")
raw = raw.toDF(*[c.replace(".", "_") for c in raw.columns])

m = F.lower(F.trim("prices_merchant"))
retalhista = (F.when(m.isin("bestbuy.com", "best buy"), "Best Buy")
    .when(m == "walmart.com", "Walmart")
    .when(m.isin("amazon.com", "amazon"), "Amazon")
    .when(m == "bhphotovideo.com", "B&H Photo")
    .when(m.isin("electronic_express", "electronic express"), "Electronic Express")
    .when(m.isin("wwstereo", "world wide stereo"), "World Wide Stereo")
    .otherwise(limpa_texto(F.col("prices_merchant"))))

linhas = (raw
    .withColumn("data", F.explode(F.split("prices_dateSeen", ",")))
    .withColumn("data", F.to_date(F.to_timestamp(F.trim("data"))))
    .filter(F.col("data").between(DATA_INICIO, DATA_FIM))
    .filter(F.col("prices_currency") == "USD")
    .filter(F.lower(F.trim("prices_condition")).isin("new", "brand new"))
    .withColumn("preco_min", F.col("prices_amountMin").cast("decimal(10,2)"))
    .withColumn("preco_max", F.col("prices_amountMax").cast("decimal(10,2)"))
    .filter((F.col("preco_min") > 0) & (F.col("preco_min") <= F.col("preco_max")))
    .withColumn("retalhista", retalhista)
    .select(F.col("id").alias("product_id"), "retalhista", "data", "preco_min", "preco_max",
            (F.upper("prices_isSale") == "TRUE").alias("is_promocao_anunciada")))
print("Preços vistos em jun-dez 2017:", linhas.count(), "| sem retalhista:", linhas.filter(F.col("retalhista").isNull()).count())

repetidos = linhas.groupBy("product_id", "retalhista", "data").count().filter("count > 1").count()
print("Combinações produto × retalhista × dia com mais de um preço:", repetidos)

precos = (linhas.groupBy("product_id", "retalhista", "data").agg(
        F.min("preco_min").alias("preco_min"), F.max("preco_max").alias("preco_max"),
        F.max(F.col("is_promocao_anunciada").cast("int")).cast("boolean").alias("is_promocao_anunciada"))
    .withColumn("is_desconto", F.col("preco_min") < F.col("preco_max"))
    .withColumn("desconto_pct", F.round((F.col("preco_max") - F.col("preco_min")) / F.col("preco_max") * 100, 2).cast("decimal(5,2)")))

assert precos.count() == precos.select("product_id", "retalhista", "data").distinct().count()
nulos(precos)
gravar_silver(precos, "datafiniti_precos")
precos.groupBy("retalhista").count().orderBy(F.desc("count")).show(10, False)

Preços vistos em jun-dez 2017: 22741 | sem retalhista: 141
Combinações produto × retalhista × dia com mais de um preço: 3740
-RECORD 0--------------------
 product_id            | 0   
 retalhista            | 141 
 data                  | 0   
 preco_min             | 0   
 preco_max             | 0   
 is_promocao_anunciada | 0   
 is_desconto           | 0   
 desconto_pct          | 0   

silver.datafiniti_precos: 17861 linhas
+------------------+-----+
|retalhista        |count|
+------------------+-----+
|Best Buy          |6847 |
|B&H Photo         |4383 |
|Walmart           |1516 |
|Beach Camera      |338  |
|buydig            |203  |
|Electronic Express|153  |
|null              |141  |
|echo-and-optics   |109  |
|Focus Camera      |105  |
|gear4less         |96   |
+------------------+-----+
only showing top 10 rows



## Produtos da Datafiniti

Um registo por produto. A coluna `primaryCategories` da Datafiniti é quase sempre "Electronics", por isso não distingue
os produtos. `categories` é uma lista de etiquetas (ex.: "Headphones,Audio,Electronics,..."): a **categoria principal** é a
primeira etiqueta da lista que não é genérica (Electronics, Consumer Electronics...). É esta categoria que entra na
`mapa_categorias` do grupo.

In [8]:
GENERICAS = ["Electronics", "Consumer Electronics", "TVs & Electronics", "Electronics Features", "Home Electronics",
             "All Electronics", "Electronics & Gadgets", "Frys"]
etiquetas = F.transform(F.split(limpa_texto(F.col("categories")), ","), lambda x: F.trim(x))
especificas = F.filter(etiquetas, lambda x: ~x.isin(GENERICAS) & (x != ""))

produtos_dat = (raw
    .select(F.col("id").alias("product_id"), limpa_texto(F.col("name")).alias("nome"), limpa_texto(F.col("brand")).alias("marca"),
            F.trim("manufacturerNumber").alias("modelo"),
            F.coalesce(F.element_at(especificas, 1), F.element_at(etiquetas, 1)).alias("categoria_principal"),
            limpa_texto(F.col("primaryCategories")).alias("categoria_datafiniti"),
            limpa_texto(F.col("categories")).alias("categorias"), F.trim("asins").alias("asins"))
    .dropDuplicates(["product_id"]))
assert produtos_dat.count() == produtos_dat.select("product_id").distinct().count()
print("Produtos da Datafiniti:", produtos_dat.count())
produtos_dat.groupBy("categoria_principal").count().orderBy(F.desc("count")).show(30, truncate=False)
gravar_silver(produtos_dat, "datafiniti_produtos")

Produtos da Datafiniti: 1245
+------------------------------+-----+
|categoria_principal           |count|
+------------------------------+-----+
|Computers                     |322  |
|Headphones                    |75   |
|Stereos                       |48   |
|Audio & Video Accessories     |46   |
|Portable Bluetooth Speakers   |43   |
|Home Audio & Theater          |35   |
|TV                            |32   |
|Auto & Tires                  |27   |
|Office                        |26   |
|Photography                   |20   |
|4K Ultra HD TVs               |18   |
|Computers & Accessories       |17   |
|In-Wall & In-Ceiling Speakers |17   |
|LED & LCD TVs                 |15   |
|Digital Cameras               |14   |
|Computers/Tablets & Networking|12   |
|Camera & Photo Accessories    |11   |
|Car & Vehicle Electronics     |11   |
|Subwoofers                    |10   |
|Outdoor Speakers              |9    |
|Pro Audio                     |9    |
|Audio                         |9  

## Ligação Datafiniti ↔ Amazon

1. **asin:** a coluna `asins` da Datafiniti tem o código Amazon do produto (às vezes vários).
2. **modelo:** para os que não ligam pelo asin, mesma marca e o código do modelo (4+ caracteres) aparece no título da Amazon.

`tipo_ligacao` diz por qual das duas regras foi feita a ligação. Na Gold, os produtos ligados (direta ou indiretamente)
recebem o mesmo `id_produto_comum`. Só se ligam asins que estão em `silver.amazon_produtos`, para a ligação não apontar
para produtos que não existem na `dim_produto`.

In [9]:
amazon_todos = spark.table("silver.amazon_produtos").select("asin", "marca", "nome")

por_asin = (produtos_dat
    .withColumn("asin", F.explode(F.split("asins", ",")))
    .withColumn("asin", F.trim("asin"))
    .join(amazon_todos.select("asin"), "asin")
    .select("product_id", "asin").withColumn("tipo_ligacao", F.lit("asin")))

sem_ligacao = produtos_dat.join(por_asin, "product_id", "left_anti")
p, a = sem_ligacao.alias("p"), amazon_todos.alias("a")
por_modelo = (p.join(a, (F.lower(F.col("p.marca")) == F.lower(F.col("a.marca")))
                        & (F.length("p.modelo") >= 4)
                        & F.lower(F.col("a.nome")).contains(F.lower(F.col("p.modelo"))))
    .select("p.product_id", "a.asin").withColumn("tipo_ligacao", F.lit("modelo")))

ligacao = por_asin.union(por_modelo).dropDuplicates(["product_id", "asin"])
gravar_silver(ligacao, "datafiniti_amazon_ligacao")
spark.sql("""
    SELECT tipo_ligacao, COUNT(DISTINCT product_id) AS produtos_datafiniti, COUNT(*) AS pares
    FROM silver.datafiniti_amazon_ligacao GROUP BY tipo_ligacao
""").show()
print("Produtos da Datafiniti sem par na Amazon:", produtos_dat.join(ligacao, "product_id", "left_anti").count())

silver.datafiniti_amazon_ligacao: 479 linhas
+------------+-------------------+-----+
|tipo_ligacao|produtos_datafiniti|pares|
+------------+-------------------+-----+
|      modelo|                 21|   23|
|        asin|                444|  456|
+------------+-------------------+-----+

Produtos da Datafiniti sem par na Amazon: 780


## Resultado

In [10]:
for t in ["amazon_avaliacoes", "amazon_produtos", "datafiniti_precos", "datafiniti_produtos", "datafiniti_amazon_ligacao"]:
    print(f"silver.{t}: {spark.table(f'silver.{t}').count()}")
spark.sql("""
    SELECT date_format(data, 'yyyy-MM') AS mes, COUNT(*) AS precos, ROUND(AVG(desconto_pct), 2) AS desconto_medio_pct
    FROM silver.datafiniti_precos GROUP BY 1 ORDER BY 1
""").show()
spark.sql("""
    SELECT date_format(data_avaliacao, 'yyyy-MM') AS mes, COUNT(*) AS avaliacoes, ROUND(AVG(nota), 2) AS nota_media
    FROM silver.amazon_avaliacoes GROUP BY 1 ORDER BY 1
""").show()

silver.amazon_avaliacoes: 1546461
silver.amazon_produtos: 172168
silver.datafiniti_precos: 17861
silver.datafiniti_produtos: 1245
silver.datafiniti_amazon_ligacao: 479
+-------+------+------------------+
|    mes|precos|desconto_medio_pct|
+-------+------+------------------+
|2017-06|  1241|              1.95|
|2017-07|  4051|              3.49|
|2017-08|  5174|              4.28|
|2017-09|  2262|              4.11|
|2017-10|  1925|              5.46|
|2017-11|  1276|              8.14|
|2017-12|  1932|              8.86|
+-------+------+------------------+

+-------+----------+----------+
|    mes|avaliacoes|nota_media|
+-------+----------+----------+
|2017-06|    225755|      4.03|
|2017-07|    234437|      4.02|
|2017-08|    253124|      4.06|
|2017-09|    225384|      4.05|
|2017-10|    203667|      4.01|
|2017-11|    198434|      4.01|
|2017-12|    205660|      3.99|
+-------+----------+----------+



In [11]:
spark.stop()